## 8. Part 9: main statistical relationship (Tân)

**Method (decision 5):** compare groups using the **median** (with Q1 and Q3) and the **outlier rate** as the main result, and add a **simple correlation** as a supporting result. The median is used because price per guest is strongly right-skewed.

**Variables (decisions 1 to 4, from Section 7):**
- Price: `price_per_guest` (USD per guest per night).
- Comparison group: `neighbourhood_cleansed` × `room_type`. Groups need at least 20 listings to be shown, and at least 100 listings when a ranking is read (decision 3).
- `outlier_rate_%`: share of listings above their group fence (all listings in the group).
- `flagged_rate_%`: share of flagged listings among listings with a known bedroom count, because a listing with no bedroom count cannot be flagged (`not enough data`).
- Correlation: Spearman rank correlation (robust to skew and outliers) between price and amenities or size.

**Reading rule:** every result describes an association in this one snapshot. None of it shows that a variable causes a higher or lower price. This section requires `p` from Section 7.

### 8.1 Groups compared: room type

In [ ]:
def compare(g):
    known = g[g["status"] != "not enough data"]          # listings with a bedroom count
    return pd.Series({
        "listings": len(g),
        "q1_ppg": g["price_per_guest"].quantile(0.25),
        "median_ppg": g["price_per_guest"].median(),
        "q3_ppg": g["price_per_guest"].quantile(0.75),
        "above_fence": int(g["above_fence"].sum()),
        "outlier_rate_%": g["above_fence"].mean() * 100,
        "listings_with_bedrooms": len(known),
        "flagged": int((g["status"] == "flagged").sum()),
        "flagged_rate_%": (known["status"] == "flagged").mean() * 100 if len(known) else np.nan,
    })

T9_room_type = p.groupby("room_type").apply(compare, include_groups=False).reset_index()
T9_room_type["missing_bedrooms_%"] = (1 - T9_room_type["listings_with_bedrooms"] / T9_room_type["listings"]) * 100
T9_room_type.round(2)

### 8.2 Groups compared: area × room type

In [ ]:
T9_group = p.groupby(GROUP_KEYS).apply(compare, include_groups=False).reset_index()
T9_group = T9_group[T9_group["listings"] >= MIN_GROUP].reset_index(drop=True)     # decision 3: at least 20
big = T9_group[T9_group["listings"] >= 100]                                         # decision 3: at least 100 to read a ranking
print(len(T9_group), "groups with at least 20 listings;", len(big), "with at least 100 "
      f"(they hold {int(big['listings'].sum()):,} of {len(p):,} listings)")
print("outlier rate across the groups with 100+ listings:", big["outlier_rate_%"].describe().round(2).to_dict())

cols = ["neighbourhood_cleansed", "room_type", "listings", "median_ppg", "outlier_rate_%", "flagged", "flagged_rate_%"]
print("\nHighest median price per guest (groups with 100+ listings)")
display(big.sort_values("median_ppg", ascending=False)[cols].head(8).round(2))
print("Highest outlier rate (groups with 100+ listings)")
display(big.sort_values("outlier_rate_%", ascending=False)[cols].head(8).round(2))
print("Groups with 100+ listings and at most 5 flagged listings:", int((big["flagged"] <= 5).sum()), "of", len(big))
print("Spearman, median price vs outlier rate across these groups:",
      round(big[["median_ppg", "outlier_rate_%"]].corr(method="spearman").iloc[0, 1], 3))

### 8.3 Simple correlation (Spearman)

Three pairs, each for all listings and inside each room type. `rho_without_flagged` repeats the result after removing the flagged listings (the check promised in the cleaning log). With this many listings even a tiny rho is statistically significant, so the **size** of rho is what matters, not a p-value.

In [ ]:
q = p.assign(amenities_vs_peers=p["amenities_count"] - p["peer_median_amenities"])
pairs = [("price_per_guest vs amenities_count", "price_per_guest", "amenities_count"),
         ("price_per_guest vs bedrooms", "price_per_guest", "bedrooms"),
         ("price_gap_pct vs amenities_vs_peers", "price_gap_pct", "amenities_vs_peers")]

def spearman(df, a, b):
    s = df[[a, b]].dropna()
    return len(s), s[a].corr(s[b], method="spearman")

rows = []
scopes = [("All listings", q)] + [(rt, q[q["room_type"] == rt]) for rt in sorted(q["room_type"].unique())]
for scope, sub in scopes:
    for label, a, b in pairs:
        n, r = spearman(sub, a, b)
        n2, r2 = spearman(sub[sub["status"] != "flagged"], a, b)
        rows.append((scope, label, n, r, n2, r2))
T9_corr = pd.DataFrame(rows, columns=["scope", "pair", "n", "spearman_rho", "n_without_flagged", "rho_without_flagged"])
T9_corr.round(3)

### 8.4 Interpretation (from this run)

**Group comparison (main result)**
- **Room types differ only a little in typical price.** Median price per guest is 34.91 USD (entire homes), 39.44 (private rooms), 43.35 (hotel rooms) and 30.00 (shared rooms). Hotel rooms (178 listings) and shared rooms (60) are low-data groups, so their medians and rates move easily.
- **Outlier rates are low and similar for the two large room types:** 1.57% for entire homes and 1.36% for private rooms. Shared rooms show 11.67%, but that is 7 listings. The flagged rate (among listings with a bedroom count) is 0.94% for entire homes and 1.34% for private rooms.
- **Areas differ more than room types.** Among the 86 groups with 100 or more listings (41,993 listings), the outlier rate has a median of 1.48% but ranges from 0% to 13.84%. Median price per guest is highest for private rooms in Favignana (81.75 USD) and in the area listed as "Cefal" in the data (74.00), and for entire homes in Taormina (61.88).
- **Rankings of flagged rates are fragile.** 67 of the 86 groups have 5 or fewer flagged listings, so one listing changes a group's rate by several percentage points. Read the ranking as a place to look more closely, not as a final order.
- **A group's typical price and its outlier rate are not related across groups:** the Spearman correlation between a group's median price and its outlier rate is -0.10.

**Correlation (supporting result)**
- **Amenities and price per guest are only weakly related.** Across all listings rho is 0.05 (0.08 for entire homes, 0.02 for private rooms). More amenities go with a slightly higher price per guest, but the link is too weak to explain prices.
- **Price per guest falls slightly with more bedrooms** (rho -0.03 overall). It is stronger for private rooms (-0.36) and shared rooms (-0.38), but these rest on small or incomplete data (see limitations).
- **Relative to its group, having more amenities than peers goes with a small positive price gap:** rho is 0.10 for entire homes and about 0 for private rooms.
- **Removing flagged listings barely changes any result:** rho moves by 0.02 or less for all listings and for the two large room types, as expected because flagged listings are about 1% of the data.

### 8.5 Limitations

1. **Association only.** The correlations and group differences do not show that amenities, size or area cause price differences. Hosts set their own prices, and other factors such as location detail, season, quality or host behaviour are not in the data.
2. **A flag marks an unusual price within a group, not proof of overpricing.** It depends on the fence rule (Q3 + 1.5 × IQR on log price), the group definition and the amenities rule.
3. **Missing bedrooms limit the flagged rate.** 65.1% of private rooms and 49.4% of hotel rooms have no bedroom count, against 2.2% of entire homes. The flagged rate for private rooms uses only 3,663 of 10,495 listings, and those listings may differ from the rest.
4. **`amenities_count` measures quantity, not quality or value.** A long list of basic items counts the same as a short list of premium ones.
5. **Price per guest assumes the price is shared evenly by `accommodates`.** This is a simplification, and it probably contributes to the negative link between bedrooms and price per guest.
6. **Small groups.** Groups under 20 listings use the room-type fallback and are not shown here. Groups with 20 to 99 listings are saved in the table, but their rates are not ranked.
7. **One snapshot and listed prices.** The data are listed prices on 1 to 2 July 2026, not prices guests paid, so seasonality and discounts are not visible.
8. **Spearman measures only monotonic association** and one number cannot show differences between areas inside the same room type.

### 8.6 Save outputs (summary tables only, no listing-level data)

In [ ]:
T9_room_type.round(2).to_csv(OUT_DIR / "T9_room_type_comparison.csv", index=False)
T9_group.round(2).to_csv(OUT_DIR / "T9_group_comparison.csv", index=False)
T9_corr.round(3).to_csv(OUT_DIR / "T9_correlation.csv", index=False)
print("saved T9_room_type_comparison.csv, T9_group_comparison.csv, T9_correlation.csv to", OUT_DIR.resolve())